# TÁI HIỆN FRAMEWORK DỰ BÁO TIÊU THỤ NĂNG LƯỢNG – THỰC NGHIỆM TRÊN `KAG_energydata_complete.csv`

**Bài báo tham chiếu:** *Innovative framework for accurate and transparent forecasting of energy consumption: A fusion of feature selection and interpretable machine learning* (Applied Energy, 2024).

**Mục tiêu notebook:** giữ lại cấu trúc phương pháp của bài báo, dùng dataset do người học cung cấp, sau đó so sánh với các baseline phổ biến. Notebook này là một môi trường thực nghiệm mới, không đọc, import, copy hay phụ thuộc vào bất kỳ notebook cũ nào.

> **Quy tắc thuật ngữ:** Markdown giải thích bằng tiếng Việt; tên riêng của phương pháp, model, thư viện, class, hàm và biến được giữ nguyên. Không dịch `mRMR`, `Random Forest`, `Relief`, `F-test FS`, `Ensemble filter`, `SIFE`, `LASSO`, `SVR-Linear`, `SVR-RBF`, `GPR`, `LSBoost`, `SHAP`, `Bayesian Optimization`, `Appliances`, `date` hoặc tên biến trong code.

## Kết quả cần tạo

1. Kiểm tra và mô tả dataset, tạo bài toán dự báo `Appliances(t+1)` từ thông tin có tại thời điểm `t`.
2. Tái hiện ba nhánh Feature Selection của bài báo: `Ensemble filter (S1)`, `Wrapper (S2)` và `Ensemble filter-Wrapper (S1S2)`.
3. Đánh giá năm model của bài báo: `LASSO`, `SVR-Linear`, `SVR-RBF`, `GPR`, `LSBoost`.
4. So sánh thêm `S0` (không Feature Selection) và các baseline thông dụng.
5. Chọn model tốt nhất trên tập test để phân tích `SHAP`.
6. Lưu dữ liệu trung gian, feature sets, metrics, predictions, plots, cấu hình và log vào Google Drive để không mất khi Colab ngắt phiên.

## Phân biệt “tái hiện phương pháp” và “tái hiện kết quả”

Ta **không** dùng dataset UK của bài báo và cũng không kỳ vọng chỉ số tuyệt đối giống Table 7. Dataset này có tần suất 10 phút, còn bài báo sử dụng dữ liệu năng lượng quốc gia theo tháng. Vì vậy mục tiêu là kiểm tra liệu *framework* Feature Selection + ML của bài báo có giúp dự báo tốt hơn các cách làm thông thường trên dữ liệu của ta hay không; không so sánh trực tiếp RMSE/MAPE tuyệt đối giữa hai dataset khác đơn vị và tần suất.

## 0. Nguyên tắc tái hiện, ánh xạ và giới hạn cần ghi trong báo cáo

### 0.1. Phần được giữ theo bài báo

- Quy trình tổng thể: preprocessing → correlation analysis → Feature Selection → Bayesian Optimization / wrapper → so sánh các feature subsets → model evaluation → `SHAP`.
- Bốn base FS methods của `Ensemble filter`: `mRMR`, `Random Forest`, `Relief`, `F-test FS`.
- Ba nhánh: `Ensemble filter (S1)`, `Wrapper (S2)`, `Ensemble filter-Wrapper (S1S2)`.
- Năm model theo phần thực nghiệm và các bảng kết quả của bài báo: `LASSO`, `SVR-Linear`, `SVR-RBF`, `GPR`, `LSBoost`.
- Tỉ lệ train/test 80/20; mục tiêu của `Wrapper` cân bằng prediction error và tỷ lệ features được chọn với trọng số `w1=0.80`, `w2=0.20`.
- `Ensemble filter`: mỗi base FS method xếp hạng features; tổng hợp relative importance và votes; ưu tiên các feature được ít nhất hai base methods bỏ phiếu.

### 0.2. Ánh xạ bắt buộc vì dataset khác bài báo

| Thành phần | Bài báo | Dataset hiện tại | Cách xử lý trong notebook |
|---|---|---|---|
| Target | `EC(t+1)` | `Appliances` | Dự báo `Appliances(t+1)` từ dữ liệu thời điểm `t` |
| Tần suất | Monthly | 10 phút | Giữ nguyên chuỗi 10 phút; một bước dự báo = 10 phút |
| Temporal features | `Mon#`, `SinM`, `CosM` | Có cả chu kỳ ngày/tuần | Giữ month encoding và bổ sung hour/day-of-week cyclic encoding; đánh dấu là adaptation |
| Split | 80/20 | Chuỗi có thứ tự thời gian | Chia chronological 80/20 để tránh dùng tương lai huấn luyện quá khứ; đây là adaptation thận trọng |
| Relief | Bài báo mô tả `Relief` theo nearest hit/miss | Target là continuous | Dùng `RReliefF`, biến thể cho regression; không dùng `ReliefF` classification sai kiểu target |
| Scaling | Min–Max normalization | Nhiều thang đo cảm biến | Fit `MinMaxScaler` trên train בלבד, áp dụng cho validation/test |
| `SIFE` | SIFE, MATLAB và tham số trong bài báo | Chạy Python/Colab | Python port minh bạch từ mô tả công bố; không tuyên bố bit-for-bit tương đương mã MATLAB gốc |

### 0.3. Hai điểm không được diễn giải sai

1. Phần Introduction của bài báo có nhắc `MLR`, nhưng phần mô tả ML, Table 3, Table 6, Table 7 và kết quả thực nghiệm sử dụng `LASSO`. Notebook đi theo **model thực nghiệm được dùng trong các bảng kết quả: `LASSO`**, không tự ý thay bằng `MLR`.
2. Bài báo không cung cấp đầy đủ source code MATLAB cho toàn bộ framework; `SIFE` được trích dẫn từ một nghiên cứu khác. Vì thế Python port ở notebook chỉ tái hiện những thành phần được mô tả rõ (set-based encoding, union/intersection/complement, fuzzy/diversity proxy, objective function và tham số đã công bố). Khi viết báo cáo, phải gọi đúng là **Python port theo mô tả công bố**, không khẳng định là mã gốc của tác giả.

### 0.4. Cách tổ chức nội dung theo báo cáo mẫu

Bố cục thực nghiệm đi theo mạch báo cáo kỹ thuật trong khóa luận mẫu: giới thiệu dữ liệu → thu thập/kiểm tra → tiền xử lý và tạo đặc trưng → tổ chức train/test → phương pháp đánh giá → kịch bản thực nghiệm → kết quả và phân tích. Chỉ tham khảo **cách tổ chức trình bày**; không lấy định nghĩa churn, OP/CP, AUC hay các thuật toán phân lớp của khóa luận để áp dụng nhầm vào bài toán hồi quy chuỗi thời gian này.

### 0.5. Trình tự chạy

- Chạy các cell từ trên xuống dưới trong runtime mới.
- Lần đầu dùng `EXPERIMENT_MODE = "quick"` để kiểm tra luồng và lỗi kỹ thuật.
- Khi mọi cell chạy ổn, đổi sang `EXPERIMENT_MODE = "paper"`, Restart runtime và Run all để tạo kết quả cuối. `quick` chỉ là smoke test, không dùng làm kết quả kết luận.
- Không mở hoặc import notebook cũ. Chỉ sử dụng CSV mà bạn chỉ định ở cell cấu hình.

## 1. Cấu trúc thực nghiệm và vai trò của từng feature subset

Notebook dùng các ký hiệu tương ứng với Figure 3 của bài báo:

- `S0`: toàn bộ candidate features, không chạy Feature Selection. Đây là control/baseline bổ sung để định lượng lợi ích của Feature Selection.
- `S1`: top `m` features từ `Ensemble filter`.
- `S2`: feature subset do `SIFE` Wrapper lựa chọn riêng cho từng model.
- `S1S2`: giao của `S1` và `S2` cho từng model, theo cách diễn giải hybrid filter-wrapper trong framework.

Với mỗi subset, huấn luyện và đánh giá năm model của bài báo. Các baseline phổ biến (`Persistence`, `Ridge`, `RandomForest`, `HistGradientBoosting`) được chạy riêng sau đó trên cùng split và cùng target.

**Lưu ý:** `S1S2` có thể có ít feature hơn `S2`. Nếu giao rỗng, notebook sẽ ghi nhận và dùng feature được xếp hạng cao nhất trong `S1` làm fallback tối thiểu để model có thể chạy; mọi fallback đều được lưu trong kết quả.

## 2. Chuẩn bị môi trường Google Colab và cài thư viện

Cell này cài các thư viện phục vụ đọc dữ liệu, trực quan hóa, Feature Selection, ML, Bayesian Optimization và SHAP. Không cài hoặc đọc bất kỳ notebook cũ nào.

In [ ]:
import sys, subprocess, importlib.util

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "pyarrow": "pyarrow",
    "shap": "shap",
    "skopt": "scikit-optimize",
}
missing = [pkg for module, pkg in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

## 3. Kết nối Google Drive và tạo workspace mới

Toàn bộ kết quả cần giữ lại sẽ được đặt trong một thư mục mới, độc lập với các workspace/notebook cũ. Colab runtime có thể bị reset; Google Drive là nơi lưu trữ bền vững.

Cấu trúc thư mục được tạo tự động:

```text
energy_forecasting_reproduction/
├── data/
│   ├── raw/
│   └── processed/
├── artifacts/
│   ├── checkpoints/
│   ├── feature_selection/
│   ├── models/
│   └── metrics/
├── figures/
├── logs/
├── reports/
└── config/
```

Nếu chưa có file CSV trong `data/raw/`, cell tiếp theo sẽ mở hộp thoại upload. Chọn đúng `KAG_energydata_complete.csv`.

In [ ]:
from pathlib import Path
import os, shutil, json, time, platform, warnings, random

try:
    from google.colab import drive, files
    IN_COLAB = True
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except Exception:
    IN_COLAB = False
    files = None
    DRIVE_ROOT = Path.cwd() / "drive_fallback"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT_NAME = "energy_forecasting_reproduction"
PROJECT_ROOT = DRIVE_ROOT / PROJECT_NAME
PATHS = {
    "raw": PROJECT_ROOT / "data" / "raw",
    "processed": PROJECT_ROOT / "data" / "processed",
    "checkpoints": PROJECT_ROOT / "artifacts" / "checkpoints",
    "feature_selection": PROJECT_ROOT / "artifacts" / "feature_selection",
    "models": PROJECT_ROOT / "artifacts" / "models",
    "metrics": PROJECT_ROOT / "artifacts" / "metrics",
    "figures": PROJECT_ROOT / "figures",
    "logs": PROJECT_ROOT / "logs",
    "reports": PROJECT_ROOT / "reports",
    "config": PROJECT_ROOT / "config",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

DATA_PATH = PATHS["raw"] / "KAG_energydata_complete.csv"
print("Project root:", PROJECT_ROOT)
print("Running in Colab:", IN_COLAB)
print("CSV exists:", DATA_PATH.exists())

In [ ]:
if not DATA_PATH.exists():
    if IN_COLAB:
        print("Upload đúng file KAG_energydata_complete.csv")
        uploaded = files.upload()
        matching = [name for name in uploaded if name.lower() == "kag_energydata_complete.csv"]
        if not matching:
            raise FileNotFoundError("Không tìm thấy KAG_energydata_complete.csv trong file upload.")
        shutil.copy2(matching[0], DATA_PATH)
    else:
        raise FileNotFoundError(
            f"Hãy đặt CSV tại: {DATA_PATH}\n"
            "Notebook không tự tìm dữ liệu trong các notebook/workspace cũ."
        )

assert DATA_PATH.exists(), "CSV chưa sẵn sàng."
print(f"Dataset source: {DATA_PATH}")
print(f"Dataset size: {DATA_PATH.stat().st_size / (1024**2):.2f} MB")

## 4. Khai báo cấu hình và lưu cấu hình thực nghiệm

`quick` giúp kiểm tra pipeline với ngân sách tính toán nhỏ. `paper` đặt lại các tham số được công bố cho `SIFE` (population size 120, max iterations 60, crossover probability 0.6, mutation probability 0.5), đồng thời tăng ngân sách cho Bayesian Optimization và các bước lấy mẫu tính toán.

**Không dùng kết quả `quick` làm kết quả chính trong báo cáo.** Các tham số của model trong Table 3 được lưu thành cấu hình tham chiếu. Do khác dataset và khác thư viện (MATLAB so với scikit-learn), một số model phải được ánh xạ sang estimator Python tương đương gần nhất; notebook ghi rõ các điểm không đồng nhất.

In [ ]:
import numpy as np
import pandas as pd

EXPERIMENT_MODE = "quick"  # Đổi thành "paper" trước lần chạy cuối.
RANDOM_SEED = 42
TEST_SIZE = 0.20
TARGET_RAW = "Appliances"
TIME_COL = "date"
N_FILTER_FEATURES = 10       # Bài báo dùng 10 features cho S1.
MIN_VOTES = 2                # Bài báo yêu cầu tối thiểu 2 votes.

if EXPERIMENT_MODE == "quick":
    SIFE_POPULATION = 12
    SIFE_ITERATIONS = 3
    SIFE_EVAL_MAX_ROWS = 3500
    RRELIEFF_UPDATES = 1200
    BAYES_ITERATIONS = 5
    BAYES_CV_SPLITS = 2
    SHAP_BACKGROUND_ROWS = 60
    SHAP_EXPLAIN_ROWS = 80
elif EXPERIMENT_MODE == "paper":
    SIFE_POPULATION = 120
    SIFE_ITERATIONS = 60
    SIFE_EVAL_MAX_ROWS = 10000
    RRELIEFF_UPDATES = 4000
    BAYES_ITERATIONS = 20
    BAYES_CV_SPLITS = 3
    SHAP_BACKGROUND_ROWS = 150
    SHAP_EXPLAIN_ROWS = 250
else:
    raise ValueError("EXPERIMENT_MODE chỉ nhận 'quick' hoặc 'paper'.")

CONFIG = {
    "project_name": PROJECT_NAME,
    "experiment_mode": EXPERIMENT_MODE,
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "target_raw": TARGET_RAW,
    "time_col": TIME_COL,
    "forecast_horizon_steps": 1,
    "sampling_interval_minutes": 10,
    "n_filter_features": N_FILTER_FEATURES,
    "minimum_votes": MIN_VOTES,
    "SIFE_population": SIFE_POPULATION,
    "SIFE_iterations": SIFE_ITERATIONS,
    "SIFE_crossover_probability": 0.6,
    "SIFE_mutation_probability": 0.5,
    "SIFE_w1_error": 0.80,
    "SIFE_w2_subset_size": 0.20,
    "RReliefF_updates": RRELIEFF_UPDATES,
    "Bayesian_Optimization_iterations": BAYES_ITERATIONS,
    "Bayesian_Optimization_cv_splits": BAYES_CV_SPLITS,
}
with open(PATHS["config"] / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 5. Import thư viện và thiết lập reproducibility

Các hàm và biến được đặt tên theo vai trò kỹ thuật. Các bước có tính ngẫu nhiên đều nhận `RANDOM_SEED`. Khi báo cáo kết quả, lưu lại phiên bản thư viện cùng cấu hình đã chạy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn
import shap

from datetime import datetime
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_regression, f_regression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.inspection import permutation_importance
from scipy.stats import randint, loguniform

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("SHAP:", shap.__version__)

## 6. Đọc dataset và kiểm tra chất lượng dữ liệu

Ở bước này chỉ đọc CSV được chỉ định. Chưa xóa dòng hay thay đổi giá trị. Ta kiểm tra schema, kiểu dữ liệu, missing values, duplicate rows, duplicate timestamps, thứ tự thời gian và sampling interval trước khi quyết định preprocessing.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

if TIME_COL not in df_raw.columns:
    raise KeyError(f"Không tìm thấy cột thời gian '{TIME_COL}'. Columns: {df_raw.columns.tolist()}")
if TARGET_RAW not in df_raw.columns:
    raise KeyError(f"Không tìm thấy target '{TARGET_RAW}'.")

df_raw[TIME_COL] = pd.to_datetime(df_raw[TIME_COL], errors="coerce")
df_raw = df_raw.dropna(subset=[TIME_COL]).sort_values(TIME_COL).reset_index(drop=True)

print("Shape:", df_raw.shape)
print("Date range:", df_raw[TIME_COL].min(), "->", df_raw[TIME_COL].max())
display(df_raw.head())
display(df_raw.dtypes.rename("dtype").to_frame())

In [ ]:
missing_table = df_raw.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame()
missing_table["missing_pct"] = 100 * missing_table["missing_count"] / len(df_raw)

row_duplicates = int(df_raw.duplicated().sum())
time_duplicates = int(df_raw[TIME_COL].duplicated().sum())
intervals = df_raw[TIME_COL].diff().dropna()
interval_counts = intervals.value_counts().head(10)

quality_summary = {
    "rows": int(len(df_raw)),
    "columns": int(df_raw.shape[1]),
    "missing_cells": int(df_raw.isna().sum().sum()),
    "duplicate_rows": row_duplicates,
    "duplicate_timestamps": time_duplicates,
    "timestamp_monotonic_increasing": bool(df_raw[TIME_COL].is_monotonic_increasing),
    "most_common_interval": str(interval_counts.index[0]) if len(interval_counts) else None,
    "interval_unique_count": int(intervals.nunique()),
}

display(missing_table)
print(json.dumps(quality_summary, ensure_ascii=False, indent=2))
print("Most common time intervals:")
display(interval_counts.rename("count").to_frame())

with open(PATHS["reports"] / "data_quality_summary.json", "w", encoding="utf-8") as f:
    json.dump(quality_summary, f, ensure_ascii=False, indent=2)
missing_table.to_csv(PATHS["reports"] / "missing_values.csv")

### 6.1. Xử lý cột trùng và dữ liệu thời gian

Kiểm tra dataset cho thấy `rv1` và `rv2` có thể trùng nhau hoàn toàn. Chỉ loại `rv2` nếu chúng thực sự giống nhau ở tất cả các dòng; không tự ý loại các sensor khác chỉ vì tương quan cao. Dữ liệu được sắp xếp theo `date`, không nội suy vì bước kiểm tra ban đầu cho thấy chuỗi cách đều 10 phút và không có missing values.

In [ ]:
df = df_raw.copy()

if "rv1" in df.columns and "rv2" in df.columns:
    rv_identical = bool(df["rv1"].equals(df["rv2"]))
else:
    rv_identical = False

if rv_identical:
    df = df.drop(columns=["rv2"])
    print("Đã loại rv2 vì trùng tuyệt đối với rv1.")
else:
    print("Không loại rv2: hai cột không trùng tuyệt đối hoặc không cùng tồn tại.")

# Dữ liệu thời gian bị trùng không được âm thầm gộp vì chưa có quy tắc aggregation từ bài báo.
if df[TIME_COL].duplicated().any():
    raise ValueError("Có duplicate timestamp. Cần xác định quy tắc xử lý trước khi tiếp tục.")

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
if df[numeric_columns].isna().any().any():
    raise ValueError("Numeric columns chứa missing values; cần phân tích nguyên nhân trước khi impute.")

print("Shape after safe duplicate-column check:", df.shape)
print("rv1/r2 identical:", rv_identical)

## 7. Khám phá target và mô hình hóa chuỗi thời gian

Target là `Appliances`. Dataset có 19.735 quan sát và khoảng lấy mẫu 10 phút. Bài toán được định nghĩa là:

\[
X_t = \{\text{sensor, weather, time features, Appliances}_t\}, \\
y_t = \text{Appliances}_{t+1}
\]

Như vậy model dự báo mức tiêu thụ ở thời điểm kế tiếp (sau 10 phút) dựa trên thông tin đang có tại thời điểm hiện tại. `Appliances_t` là lag feature hợp lệ, không phải target cùng thời điểm. Hàng cuối cùng không có nhãn `t+1` nên sẽ bị loại.

In [ ]:
target_stats = df[TARGET_RAW].describe().to_frame("Appliances")
display(target_stats)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(df[TIME_COL], df[TARGET_RAW], linewidth=0.7)
axes[0].set_title("Appliances theo thời gian")
axes[0].set_xlabel("date")
axes[0].set_ylabel("Appliances")

sns.histplot(df[TARGET_RAW], bins=50, kde=True, ax=axes[1])
axes[1].set_title("Phân phối target Appliances")
plt.tight_layout()
plt.savefig(PATHS["figures"] / "target_overview.png", dpi=160, bbox_inches="tight")
plt.show()

lag_corr = pd.Series({
    "lag_1": df[TARGET_RAW].autocorr(1),
    "lag_6": df[TARGET_RAW].autocorr(6),
    "lag_144": df[TARGET_RAW].autocorr(144),
}, name="autocorrelation")
display(lag_corr.to_frame())

## 8. Tạo supervised dataset và temporal features

Bài báo dùng `Mon#`, `SinM`, `CosM` để thể hiện chu kỳ tháng. Với dữ liệu 10 phút, biến thiên trong ngày có thể quan trọng hơn chu kỳ tháng; vì thế notebook giữ nguyên month encoding và bổ sung chu kỳ giờ (`SinH`, `CosH`) và ngày trong tuần (`SinDOW`, `CosDOW`). Đây là **adaptation theo sampling frequency**, không phải nói rằng bài báo đã sử dụng các biến giờ/ngày này.

Không đưa cột `date` dạng timestamp thô vào model. Tất cả biến dự báo ở thời điểm `t`; target được shift sang bước kế tiếp.

In [ ]:
supervised = df.copy()

dt = supervised[TIME_COL]
supervised["Appliances_t"] = supervised[TARGET_RAW]

# Month cyclic encoding: giữ đúng ý tưởng Mon#, SinM, CosM của bài báo.
supervised["Mon#"] = dt.dt.month.astype(int)
supervised["SinM"] = np.sin(2 * np.pi * (dt.dt.month - 1) / 12)
supervised["CosM"] = np.cos(2 * np.pi * (dt.dt.month - 1) / 12)

# Cyclic features bổ sung cho chu kỳ 10 phút.
minute_of_day = dt.dt.hour * 60 + dt.dt.minute
supervised["Hour#"] = dt.dt.hour.astype(int)
supervised["SinH"] = np.sin(2 * np.pi * minute_of_day / (24 * 60))
supervised["CosH"] = np.cos(2 * np.pi * minute_of_day / (24 * 60))

supervised["DOW#"] = dt.dt.dayofweek.astype(int)
supervised["SinDOW"] = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
supervised["CosDOW"] = np.cos(2 * np.pi * dt.dt.dayofweek / 7)

# One-step-ahead label (10 phút tiếp theo).
supervised["Appliances_t_plus_1"] = supervised[TARGET_RAW].shift(-1)
supervised = supervised.dropna(subset=["Appliances_t_plus_1"]).reset_index(drop=True)

# Không dùng target cùng thời điểm làm input; chỉ dùng lag Appliances_t.
excluded_columns = {TIME_COL, TARGET_RAW, "Appliances_t_plus_1"}
feature_columns = [c for c in supervised.columns if c not in excluded_columns]

X_all = supervised[feature_columns].select_dtypes(include=np.number).copy()
y_all = supervised["Appliances_t_plus_1"].astype(float).copy()
time_all = supervised[TIME_COL].copy()

assert "Appliances_t" in X_all.columns
assert TARGET_RAW not in X_all.columns
assert "Appliances_t_plus_1" not in X_all.columns
assert len(X_all) == len(y_all)

print("Supervised shape:", X_all.shape)
print("Target:", "Appliances_t_plus_1")
print("Number of candidate features:", len(feature_columns))
print("Features:", feature_columns)
print("Time range:", time_all.min(), "->", time_all.max())

display(supervised[[TIME_COL, "Appliances_t", "Appliances_t_plus_1"]].head(8))
supervised.to_parquet(PATHS["processed"] / "supervised_dataset.parquet", index=False)

## 9. Chia train/test theo thời gian: 80/20

Bài báo ghi tỉ lệ 80/20. Vì dataset hiện tại là chuỗi thời gian, notebook không xáo trộn ngẫu nhiên: 80% phần đầu dùng cho huấn luyện và lựa chọn mô hình; 20% phần cuối giữ nguyên làm holdout test. Tất cả việc tính feature importance, scaling, Bayesian Optimization và Wrapper Feature Selection chỉ được phép nhìn thấy train.

Đây là biện pháp ngăn data leakage cho chuỗi thời gian. Không dùng test set để chọn feature hay điều chỉnh hyperparameter.

In [ ]:
split_idx = int(len(X_all) * (1 - TEST_SIZE))
X_train_raw = X_all.iloc[:split_idx].copy()
X_test_raw = X_all.iloc[split_idx:].copy()
y_train_raw = y_all.iloc[:split_idx].copy()
y_test_raw = y_all.iloc[split_idx:].copy()
time_train = time_all.iloc[:split_idx].copy()
time_test = time_all.iloc[split_idx:].copy()

# Fit scaler chỉ trên train để tránh leakage.
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = pd.DataFrame(
    scaler_X.fit_transform(X_train_raw),
    columns=feature_columns,
    index=X_train_raw.index,
)
X_test = pd.DataFrame(
    scaler_X.transform(X_test_raw),
    columns=feature_columns,
    index=X_test_raw.index,
)
y_train = pd.Series(
    scaler_y.fit_transform(y_train_raw.to_numpy().reshape(-1, 1)).ravel(),
    index=y_train_raw.index,
    name="y_scaled",
)
y_test = y_test_raw.copy()  # Giữ target gốc để đánh giá chỉ số theo đơn vị dataset.

SPLIT_INFO = {
    "split_type": "chronological",
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_start": str(time_train.min()),
    "train_end": str(time_train.max()),
    "test_start": str(time_test.min()),
    "test_end": str(time_test.max()),
    "test_fraction": float(TEST_SIZE),
}
with open(PATHS["config"] / "split_info.json", "w", encoding="utf-8") as f:
    json.dump(SPLIT_INFO, f, ensure_ascii=False, indent=2)

print(json.dumps(SPLIT_INFO, ensure_ascii=False, indent=2))
joblib.dump({"scaler_X": scaler_X, "scaler_y": scaler_y, "feature_columns": feature_columns},
            PATHS["checkpoints"] / "scalers_and_feature_columns.joblib")

## 10. Correlation analysis

Bài báo sử dụng Pearson correlation để phân tích quan hệ tuyến tính giữa target và input features. Để giữ đúng nguyên tắc chống leakage, heatmap này được tính trên training subset, không dùng test. Correlation không tự động là lý do xóa feature; quyết định chọn feature sẽ do ba nhánh FS đảm nhiệm.

In [ ]:
train_for_corr = X_train.copy()
train_for_corr["target"] = y_train.to_numpy()
corr_matrix = train_for_corr.corr(method="pearson")

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Pearson correlation – training data only")
plt.tight_layout()
plt.savefig(PATHS["figures"] / "pearson_correlation_train.png", dpi=160, bbox_inches="tight")
plt.show()

corr_with_target = corr_matrix["target"].drop("target").sort_values(key=lambda x: x.abs(), ascending=False)
display(corr_with_target.to_frame("pearson_corr_with_target"))
corr_with_target.to_csv(PATHS["reports"] / "pearson_corr_with_target.csv")

## 11. Base Feature Selection methods của Ensemble filter

Bốn base methods theo bài báo:

1. `mRMR`: maximize relevance với target và giảm redundancy giữa các predictors bằng Mutual Information.
2. `Random Forest`: dùng feature importance từ `RandomForestRegressor`.
3. `Relief`: dùng `RReliefF` vì target hiện tại là continuous; phiên bản này cập nhật trọng số dựa trên độ khác biệt response và feature của các nearest neighbors.
4. `F-test FS`: dùng `f_regression` cho hồi quy.

Các method chỉ fit trên `X_train`, `y_train`. Với `mRMR`, tính Mutual Information trên sample train có giới hạn kích thước để giảm thời gian tính toán; đây là sampling kỹ thuật, không phải lấy test data.

In [ ]:
def _rank_to_score(scores, feature_names):
    s = pd.Series(np.asarray(scores, dtype=float), index=list(feature_names)).replace([np.inf, -np.inf], np.nan)
    s = s.fillna(s.min() if s.notna().any() else 0.0)
    # Chuyển thành relative rank score [0, 1], không phụ thuộc thang đo giữa các thuật toán.
    order = s.rank(method="average", ascending=True)
    if len(s) <= 1:
        normalized = pd.Series(1.0, index=s.index)
    else:
        normalized = (order - 1) / (len(s) - 1)
    return normalized.sort_values(ascending=False)


def mrmr_regression_scores(X, y, random_state=42, max_rows=5000):
    Xv = np.asarray(X, dtype=float)
    yv = np.asarray(y, dtype=float)
    n, p = Xv.shape
    rng = np.random.default_rng(random_state)
    if n > max_rows:
        idx = np.sort(rng.choice(n, size=max_rows, replace=False))
        Xv, yv = Xv[idx], yv[idx]
    relevance = mutual_info_regression(Xv, yv, random_state=random_state)
    redundancy = np.zeros((p, p), dtype=float)
    # Symmetric pairwise MI between predictors.
    for i in range(p):
        for j in range(i + 1, p):
            mi_ij = mutual_info_regression(Xv[:, [i]], Xv[:, j], random_state=random_state)[0]
            redundancy[i, j] = mi_ij
            redundancy[j, i] = mi_ij
    selected, remaining = [], set(range(p))
    scores = np.zeros(p, dtype=float)
    while remaining:
        best_idx, best_score = None, -np.inf
        for j in remaining:
            if not selected:
                score = relevance[j]
            else:
                score = relevance[j] - redundancy[j, selected].mean()
            if score > best_score:
                best_idx, best_score = j, score
        selected.append(best_idx)
        remaining.remove(best_idx)
        scores[best_idx] = -len(selected)  # Rank encoding: selected first -> larger score after conversion below.
    ranks = np.empty(p, dtype=float)
    for rank, idx in enumerate(selected, start=1):
        ranks[idx] = p - rank + 1
    return ranks


def rrelieff_regression_scores(X, y, n_updates=1200, k=10, sigma=50.0, random_state=42):
    """RReliefF cho continuous target; cập nhật từ nearest neighbors, không tạo ma trận n x n."""
    Xv = np.asarray(X, dtype=float)
    yv = np.asarray(y, dtype=float).ravel()
    n, p = Xv.shape
    if n < 3:
        return np.zeros(p)
    k = min(k, n - 1)
    rng = np.random.default_rng(random_state)
    query_n = min(n_updates, n)
    query_idx = rng.choice(n, size=query_n, replace=False)
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean", n_jobs=-1)
    nn.fit(Xv)
    distances, neighbors = nn.kneighbors(Xv[query_idx], return_distance=True)
    y_range = max(float(np.nanmax(yv) - np.nanmin(yv)), 1e-12)
    x_ranges = np.maximum(np.nanmax(Xv, axis=0) - np.nanmin(Xv, axis=0), 1e-12)
    Wdy = 0.0
    Wdj = np.zeros(p, dtype=float)
    Wdydj = np.zeros(p, dtype=float)
    effective_updates = 0

    for row, source_idx in enumerate(query_idx):
        # Loại chính instance khỏi danh sách nearest neighbors.
        neigh_idx = neighbors[row]
        neigh_dist = distances[row]
        valid = neigh_idx != source_idx
        neigh_idx = neigh_idx[valid][:k]
        neigh_dist = neigh_dist[valid][:k]
        if len(neigh_idx) == 0:
            continue
        ranks = np.arange(1, len(neigh_idx) + 1, dtype=float)
        weights = np.exp(-((ranks / sigma) ** 2))
        weights = weights / max(weights.sum(), 1e-12)
        dy = np.abs(yv[source_idx] - yv[neigh_idx]) / y_range
        dx = np.abs(Xv[source_idx] - Xv[neigh_idx]) / x_ranges
        Wdy += float(np.sum(weights * dy))
        Wdj += np.sum(weights[:, None] * dx, axis=0)
        Wdydj += np.sum(weights[:, None] * dx * dy[:, None], axis=0)
        effective_updates += 1

    if effective_updates == 0:
        return np.zeros(p)
    Wdy /= effective_updates
    Wdj /= effective_updates
    Wdydj /= effective_updates
    eps = 1e-12
    first = Wdydj / max(Wdy, eps)
    second = (Wdj - Wdydj) / max(1.0 - Wdy, eps)
    weights = first - second
    return np.nan_to_num(weights, nan=0.0, posinf=0.0, neginf=0.0)


def compute_base_fs_scores(X, y, feature_names, random_state=42):
    Xv = np.asarray(X, dtype=float)
    yv = np.asarray(y, dtype=float)
    rng = np.random.default_rng(random_state)

    mrmr_raw = mrmr_regression_scores(Xv, yv, random_state=random_state)

    rf_n = min(len(Xv), 10000 if EXPERIMENT_MODE == "quick" else 16000)
    rf_idx = np.sort(rng.choice(len(Xv), size=rf_n, replace=False)) if len(Xv) > rf_n else np.arange(len(Xv))
    rf = RandomForestRegressor(n_estimators=250, max_features=0.8, min_samples_leaf=2,
                               random_state=random_state, n_jobs=-1)
    rf.fit(Xv[rf_idx], yv[rf_idx])
    rf_raw = rf.feature_importances_

    relief_raw = rrelieff_regression_scores(
        Xv, yv, n_updates=RRELIEFF_UPDATES, k=10, sigma=50.0, random_state=random_state
    )

    f_raw, _ = f_regression(Xv, yv)
    f_raw = np.nan_to_num(f_raw, nan=0.0, posinf=np.nanmax(np.nan_to_num(f_raw, nan=0.0, posinf=0.0)), neginf=0.0)

    score_df = pd.DataFrame(index=list(feature_names))
    score_df["mRMR"] = _rank_to_score(mrmr_raw, feature_names)
    score_df["Random Forest"] = _rank_to_score(rf_raw, feature_names)
    score_df["Relief"] = _rank_to_score(relief_raw, feature_names)
    score_df["F-test FS"] = _rank_to_score(f_raw, feature_names)
    return score_df, rf

base_fs_scores, rf_selector_model = compute_base_fs_scores(
    X_train, y_train, feature_columns, random_state=RANDOM_SEED
)
display(base_fs_scores.sort_values("mRMR", ascending=False).head(15))
base_fs_scores.to_csv(PATHS["feature_selection"] / "base_fs_relative_scores.csv")
joblib.dump(rf_selector_model, PATHS["models"] / "random_forest_feature_selector.joblib")

## 12. Ensemble filter: kết hợp score và voting để tạo S1

Theo bài báo, đặt `m` là số features mong muốn và `v` là số votes tối thiểu. Mỗi base FS method lấy top `m`; sau đó các relative importance được tổng hợp bằng weighted average và vote count. Notebook dùng trọng số bằng nhau cho bốn base methods vì bài báo không đưa một bộ trọng số riêng khác trong phần mô tả thuật toán. Chọn tối đa `m=10` features, ưu tiên feature nhận ít nhất `v=2` votes.

In [ ]:
base_method_names = ["mRMR", "Random Forest", "Relief", "F-test FS"]
m = min(N_FILTER_FEATURES, len(feature_columns))

vote_counts = pd.Series(0, index=feature_columns, dtype=int)
for method in base_method_names:
    top_method = base_fs_scores[method].nlargest(m).index
    vote_counts.loc[top_method] += 1

ensemble_table = base_fs_scores.copy()
ensemble_table["Vote"] = vote_counts
ensemble_table["Average importance"] = ensemble_table[base_method_names].mean(axis=1)
ensemble_table["vote_threshold_met"] = ensemble_table["Vote"] >= MIN_VOTES

eligible = ensemble_table[ensemble_table["vote_threshold_met"]].copy()
eligible = eligible.sort_values(["Vote", "Average importance"], ascending=[False, False])

if len(eligible) >= m:
    S1 = eligible.head(m).index.tolist()
else:
    # Bổ sung theo average importance nếu không đủ feature đạt tối thiểu 2 votes.
    selected = eligible.index.tolist()
    remaining = ensemble_table.drop(index=selected).sort_values(
        ["Average importance", "Vote"], ascending=[False, False]
    ).index.tolist()
    S1 = (selected + remaining)[:m]
    ensemble_table["filled_below_vote_threshold"] = ensemble_table.index.isin(S1) & (ensemble_table["Vote"] < MIN_VOTES)

ensemble_table["Final rank"] = ensemble_table["Average importance"].rank(method="first", ascending=False).astype(int)
ensemble_table["selected_S1"] = ensemble_table.index.isin(S1)
ensemble_table = ensemble_table.sort_values(["selected_S1", "Vote", "Average importance"], ascending=[False, False, False])

print("S1 features:", S1)
print("S1 count:", len(S1))
display(ensemble_table)
ensemble_table.to_csv(PATHS["feature_selection"] / "ensemble_filter_S1.csv")
with open(PATHS["feature_selection"] / "S1_features.json", "w", encoding="utf-8") as f:
    json.dump(S1, f, ensure_ascii=False, indent=2)

## 13. Khởi tạo năm ML models theo Table 3 của bài báo

Dùng các giá trị tham chiếu được báo cáo trong Table 3 làm cấu hình khởi tạo. Bài báo dùng MATLAB R2023b; notebook dùng scikit-learn. Một số tham số có tên/cách cài đặt khác nhau giữa hai thư viện nên đây là ánh xạ gần nhất, không phải cùng implementation nội bộ.

| Model | Giá trị trong bài báo | Ánh xạ trong notebook |
|---|---|---|
| `LASSO` | lambda = 1.16e-04 | `sklearn.linear_model.Lasso(alpha=1.16e-4)` |
| `SVR-Linear` | BoxConstraint = 10; Epsilon = 0.005 | `SVR(kernel='linear', C=10, epsilon=0.005)` |
| `SVR-RBF` | BoxConstraint = 10.5; Gamma = 35; Epsilon = 0.003 | `SVR(kernel='rbf', C=10.5, gamma=35, epsilon=0.003)` |
| `GPR` | Sigma = 0.046; subset of regressors; pure quadratic basis | GaussianProcessRegressor với RBF kernel và giới hạn số training rows; khác biệt implementation được ghi nhận |
| `LSBoost` | MaxNumSplits = 12; cycles = 8; learning rate = 0.28 | GradientBoostingRegressor với `max_leaf_nodes=13`, `n_estimators=8`, `learning_rate=0.28` |

`GPR` và `LSBoost` là hai model có khác biệt đáng kể hơn khi chuyển từ MATLAB sang scikit-learn; không nên tuyên bố hai thư viện tạo ra estimator hoàn toàn giống nhau.

In [ ]:
class CappedGaussianProcess(BaseEstimator, RegressorMixin):
    """GPR Python approximation; giới hạn số training rows để tránh O(n^3) trên chuỗi lớn."""
    def __init__(self, length_scale=0.046, alpha=1e-3, max_train_size=1800, random_state=42):
        self.length_scale = length_scale
        self.alpha = alpha
        self.max_train_size = max_train_size
        self.random_state = random_state

    def fit(self, X, y):
        Xv = np.asarray(X, dtype=float)
        yv = np.asarray(y, dtype=float)
        if len(Xv) > self.max_train_size:
            idx = np.linspace(0, len(Xv) - 1, self.max_train_size).astype(int)
            Xv, yv = Xv[idx], yv[idx]
        kernel = ConstantKernel(1.0, constant_value_bounds="fixed") * RBF(
            length_scale=float(self.length_scale), length_scale_bounds="fixed"
        ) + WhiteKernel(noise_level=float(self.alpha), noise_level_bounds="fixed")
        self.model_ = GaussianProcessRegressor(
            kernel=kernel, alpha=0.0, normalize_y=False, optimizer=None,
            random_state=self.random_state
        )
        self.model_.fit(Xv, yv)
        return self

    def predict(self, X):
        return self.model_.predict(np.asarray(X, dtype=float))


def make_paper_models(mode="reference"):
    models = {
        "LASSO": Lasso(alpha=1.16e-4, max_iter=30000, random_state=RANDOM_SEED),
        "SVR-Linear": SVR(kernel="linear", C=10.0, epsilon=0.005),
        "SVR-RBF": SVR(kernel="rbf", C=10.5, gamma=35.0, epsilon=0.003),
        "GPR": CappedGaussianProcess(length_scale=0.046, alpha=1e-3,
                                      max_train_size=1800 if EXPERIMENT_MODE == "quick" else 3000,
                                      random_state=RANDOM_SEED),
        "LSBoost": GradientBoostingRegressor(
            loss="squared_error", n_estimators=8, learning_rate=0.28,
            max_leaf_nodes=13, random_state=RANDOM_SEED,
        ),
    }
    return models

paper_models = make_paper_models()
print("Paper models:", list(paper_models.keys()))

## 14. Bayesian Optimization cho hyperparameters

Bài báo sử dụng `Bayesian Optimization` để tối ưu hyperparameters. Cell này cung cấp lựa chọn tối ưu hóa trên training data bằng `BayesSearchCV` và `TimeSeriesSplit`. Tập test cuối cùng không tham gia quá trình này.

- Để kiểm tra nhanh: `RUN_BAYES_OPTIMIZATION = False` có thể tạm thời giữ các giá trị Table 3.
- `quick` mặc định tắt bước này để smoke test nhanh; `paper` mặc định bật `RUN_BAYES_OPTIMIZATION = True`. Số vòng tìm kiếm do `BAYES_ITERATIONS` điều khiển.
- `GPR` dùng estimator giới hạn training rows để khả thi với dataset lớn; đây là approximation, không phải đúng MATLAB `subset of regressors`.

In [ ]:
RUN_BAYES_OPTIMIZATION = (EXPERIMENT_MODE == "paper")  # Quick: False; paper: True.


def run_bayesian_optimization(X, y, initial_models):
    try:
        from skopt import BayesSearchCV
        from skopt.space import Real, Integer
    except Exception as exc:
        raise ImportError("Không import được scikit-optimize. Hãy chạy lại cell cài thư viện.") from exc

    search_spaces = {
        "LASSO": {"alpha": Real(1e-6, 1.0, prior="log-uniform")},
        "SVR-Linear": {
            "C": Real(0.1, 100.0, prior="log-uniform"),
            "epsilon": Real(1e-4, 0.1, prior="log-uniform"),
        },
        "SVR-RBF": {
            "C": Real(0.1, 100.0, prior="log-uniform"),
            "gamma": Real(1e-3, 100.0, prior="log-uniform"),
            "epsilon": Real(1e-4, 0.1, prior="log-uniform"),
        },
        "GPR": {
            "length_scale": Real(1e-3, 2.0, prior="log-uniform"),
            "alpha": Real(1e-5, 0.1, prior="log-uniform"),
        },
        "LSBoost": {
            "n_estimators": Integer(30, 250),
            "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
            "max_leaf_nodes": Integer(3, 32),
        },
    }

    cv = TimeSeriesSplit(n_splits=BAYES_CV_SPLITS)
    best_models, best_params, search_objects = {}, {}, {}
    for model_name, estimator in initial_models.items():
        print(f"Bayesian Optimization: {model_name}")
        cache_file = PATHS["checkpoints"] / f"bayes_search_{model_name.replace('-', '_')}.joblib"
        cache_meta_file = PATHS["checkpoints"] / f"bayes_search_{model_name.replace('-', '_')}.meta.json"
        expected_meta = {
            "model": model_name,
            "n_iter": BAYES_ITERATIONS,
            "cv_splits": BAYES_CV_SPLITS,
            "train_rows": int(len(X)),
            "feature_names": list(X.columns),
            "random_state": RANDOM_SEED,
        }
        search = None
        if cache_file.exists() and cache_meta_file.exists():
            try:
                stored_meta = json.loads(cache_meta_file.read_text(encoding="utf-8"))
                if stored_meta == expected_meta:
                    cached_search = joblib.load(cache_file)
                    if hasattr(cached_search, "best_estimator_"):
                        search = cached_search
                        print(" Loaded matching Bayesian Optimization checkpoint.")
            except Exception as exc:
                print(" Could not load Bayesian checkpoint; running search again:", repr(exc))
        if search is None:
            search = BayesSearchCV(
                estimator=estimator,
                search_spaces=search_spaces[model_name],
                n_iter=BAYES_ITERATIONS,
                scoring="neg_root_mean_squared_error",
                cv=cv,
                n_jobs=1,
                random_state=RANDOM_SEED,
                refit=True,
                error_score="raise",
            )
            search.fit(X, y)
            joblib.dump(search, cache_file)
            cache_meta_file.write_text(json.dumps(expected_meta, ensure_ascii=False, indent=2), encoding="utf-8")
        best_models[model_name] = search.best_estimator_
        best_params[model_name] = search.best_params_
        search_objects[model_name] = search
        print(" best score:", -search.best_score_, "best params:", search.best_params_)

    with open(PATHS["config"] / "bayesian_optimization_best_params.json", "w", encoding="utf-8") as f:
        json.dump(best_params, f, ensure_ascii=False, indent=2, default=str)
    return best_models, best_params, search_objects

if RUN_BAYES_OPTIMIZATION:
    paper_models, bayes_best_params, bayes_search_objects = run_bayesian_optimization(
        X_train, y_train, paper_models
    )
else:
    bayes_best_params = {name: "Table 3 reference configuration" for name in paper_models}
    bayes_search_objects = {}
    print("Giữ cấu hình tham chiếu Table 3. Bayesian Optimization chưa chạy.")

## 15. Wrapper FS bằng SIFE: objective và giới hạn tái hiện

Bài báo định nghĩa objective function:

\[
\min F(s_i)=w_1\,Er(s_i)+w_2\,Ps(s_i), \quad
w_1=0.80,\;w_2=0.20
\]

Trong đó `Er` là prediction error (RMSE hoặc MAPE) và `Ps` là tỷ lệ features được chọn. Notebook dùng RMSE trên validation segment nội bộ của training data, sau đó cộng penalty tỷ lệ features. Tập test giữ kín cho đến bước đánh giá cuối.

**Quan trọng:** SIFE đầy đủ sử dụng integer coding, three-parent crossover với set operations (`union`, `intersection`, `complement`) và fuzzy granulation. Bài báo 2024 nêu `population size=120`, `max iteration=60`, `crossover probability=0.6`, `mutation probability=0.5`, nhưng không đính kèm toàn bộ source code gốc. Python implementation bên dưới là **port theo mô tả công bố với fuzzy-diversity proxy**, không phải xác nhận tương đương tuyệt đối với implementation MATLAB của tác giả. Ghi rõ hạn chế này trong report.

In [ ]:
def _jaccard_distance(a, b):
    a, b = set(a), set(b)
    if not a and not b:
        return 0.0
    return 1.0 - len(a & b) / max(len(a | b), 1)


def _fit_predict_safe(estimator, X_fit, y_fit, X_eval, model_name=""):
    est = clone(estimator)
    Xf = np.asarray(X_fit, dtype=float)
    yf = np.asarray(y_fit, dtype=float)
    Xe = np.asarray(X_eval, dtype=float)
    if model_name == "GPR" and len(Xf) > 1200:
        idx = np.linspace(0, len(Xf) - 1, 1200).astype(int)
        Xf, yf = Xf[idx], yf[idx]
    est.fit(Xf, yf)
    return est.predict(Xe), est


def sife_python_port(estimator, X, y, feature_names, model_name="", population_size=12,
                     max_iter=3, pc=0.6, pm=0.5, w1=0.80, w2=0.20,
                     max_eval_rows=3500, random_state=42):
    """Set-based SIFE-style wrapper port. Not the original MATLAB implementation."""
    rng = np.random.default_rng(random_state)
    Xv = np.asarray(X, dtype=float)
    yv = np.asarray(y, dtype=float).ravel()
    n, p = Xv.shape
    all_features = set(range(p))
    if p == 1:
        return [feature_names[0]], pd.DataFrame([{"generation": 0, "best_objective": np.nan, "n_features": 1}])

    # Giới hạn rows của wrapper objective để notebook có thể chạy trên Colab.
    if n > max_eval_rows:
        idx = np.linspace(0, n - 1, max_eval_rows).astype(int)
        Xv, yv = Xv[idx], yv[idx]
    split = max(2, int(len(Xv) * 0.80))
    X_fit, X_val = Xv[:split], Xv[split:]
    y_fit, y_val = yv[:split], yv[split:]
    if len(X_val) == 0:
        X_fit, X_val = Xv[:-1], Xv[-1:]
        y_fit, y_val = yv[:-1], yv[-1:]

    cache = {}
    eval_counter = 0

    def evaluate(chromosome):
        nonlocal eval_counter
        chrom = tuple(sorted(set(chromosome)))
        if not chrom:
            chrom = (int(rng.integers(0, p)),)
        if chrom in cache:
            return cache[chrom]
        pred, _ = _fit_predict_safe(
            estimator, X_fit[:, chrom], y_fit, X_val[:, chrom], model_name=model_name
        )
        rmse = float(np.sqrt(mean_squared_error(y_val, pred)))
        subset_penalty = len(chrom) / p
        objective = w1 * rmse + w2 * subset_penalty
        cache[chrom] = (objective, rmse, subset_penalty)
        eval_counter += 1
        return cache[chrom]

    # Population initialization có phân tầng kích thước để duy trì diversity.
    population = []
    sizes = np.linspace(1, p, num=max(population_size, 2)).round().astype(int)
    for size in sizes:
        chromosome = tuple(sorted(rng.choice(p, size=min(max(int(size), 1), p), replace=False).tolist()))
        population.append(chromosome)
    while len(population) < population_size:
        size = int(rng.integers(1, p + 1))
        population.append(tuple(sorted(rng.choice(p, size=size, replace=False).tolist())))
    population = list(dict.fromkeys(population))
    while len(population) < population_size:
        size = int(rng.integers(1, p + 1))
        population.append(tuple(sorted(rng.choice(p, size=size, replace=False).tolist())))

    history = []
    for generation in range(max_iter + 1):
        scored = sorted([(evaluate(ch), ch) for ch in population], key=lambda z: z[0][0])
        best_metric, best_chrom = scored[0]
        history.append({
            "generation": generation,
            "best_objective": best_metric[0],
            "best_RMSE_validation": best_metric[1],
            "best_subset_penalty": best_metric[2],
            "n_features": len(best_chrom),
            "unique_evaluations": eval_counter,
        })
        if generation == max_iter:
            break

        elite_count = max(2, int(0.15 * population_size))
        next_population = [ch for _, ch in scored[:elite_count]]
        # Fuzzy-granulation proxy: ưu tiên cá thể khác biệt (Jaccard distance) so với elite.
        candidates = [ch for _, ch in scored[elite_count:]]
        candidates.sort(key=lambda ch: min((_jaccard_distance(ch, e) for e in next_population), default=1.0), reverse=True)
        for ch in candidates:
            if len(next_population) >= min(population_size, elite_count + max(1, population_size // 4)):
                break
            if min((_jaccard_distance(ch, e) for e in next_population), default=1.0) > 0.10:
                next_population.append(ch)

        while len(next_population) < population_size:
            # Tournament selection: lấy 3 parent để dùng union/intersection set operations.
            parent_candidates = rng.choice(len(scored), size=min(3, len(scored)), replace=False)
            parents = [set(scored[i][1]) for i in parent_candidates]
            while len(parents) < 3:
                parents.append(set(scored[int(rng.integers(0, len(scored)))][1]))
            A, B, C = parents
            union_ab = A | B
            intersection_ab = A & B
            if rng.random() < pc:
                child = set(intersection_ab)
                pool = sorted(union_ab - intersection_ab)
                if pool:
                    add_n = int(rng.integers(0, len(pool) + 1))
                    if add_n:
                        child.update(rng.choice(pool, size=add_n, replace=False).tolist())
                # Parent thứ ba điều tiết exploration/exploitation.
                if rng.random() < 0.5:
                    child = child | (C & union_ab)
                else:
                    child = child & (C | intersection_ab)
            else:
                child = set(rng.choice(sorted(union_ab) if union_ab else list(all_features),
                                       size=max(1, min(len(union_ab) or 1, p)), replace=False).tolist())

            # Mutation bằng complement: thêm/bỏ feature từ Ω \ child.
            if rng.random() < pm:
                complement = sorted(all_features - child)
                if complement and rng.random() < 0.5:
                    child.add(int(rng.choice(complement)))
                elif len(child) > 1:
                    child.remove(int(rng.choice(sorted(child))))
            if not child:
                child.add(int(rng.integers(0, p)))
            chromosome = tuple(sorted(child))
            if chromosome not in next_population:
                next_population.append(chromosome)
        population = next_population[:population_size]

    best = min(population, key=lambda ch: evaluate(ch)[0])
    # Population could have changed after the last generation; compare cached candidates too.
    best = min(cache.keys(), key=lambda ch: cache[ch][0])
    selected_features = [feature_names[i] for i in best]
    history_df = pd.DataFrame(history)
    history_df["model"] = model_name
    history_df["unique_evaluations_final"] = eval_counter
    return selected_features, history_df

## 16. Chạy Wrapper FS (S2) riêng cho từng paper model

`S2` được chạy model-specific vì Wrapper FS phụ thuộc model được dùng để đánh giá prediction error. Trong `quick`, population/iterations và số hàng validation bị giới hạn để kiểm tra luồng. Trong `paper`, các giá trị SIFE trong bài báo được khôi phục.

Quá trình này có thể là phần tốn thời gian nhất. Mỗi model có một checkpoint riêng; nếu Colab ngắt sau một model, bạn có thể nạp các file JSON/CSV đã được lưu trên Drive thay vì mất toàn bộ kết quả.

In [ ]:
S2_by_model = {}
sife_histories = []

for model_name, estimator in paper_models.items():
    s2_file = PATHS["feature_selection"] / f"S2_{model_name.replace('-', '_')}.json"
    history_file = PATHS["feature_selection"] / f"SIFE_history_{model_name.replace('-', '_')}.csv"
    meta_file = PATHS["feature_selection"] / f"SIFE_meta_{model_name.replace('-', '_')}.json"
    expected_meta = {
        "experiment_mode": EXPERIMENT_MODE,
        "population_size": SIFE_POPULATION,
        "max_iter": SIFE_ITERATIONS,
        "max_eval_rows": SIFE_EVAL_MAX_ROWS,
        "pc": 0.6,
        "pm": 0.5,
        "w1": 0.80,
        "w2": 0.20,
        "random_state": RANDOM_SEED,
        "model": model_name,
        "estimator_params": {k: str(v) for k, v in estimator.get_params(deep=False).items()},
    }

    checkpoint_matches = False
    if s2_file.exists() and history_file.exists() and meta_file.exists():
        try:
            stored_meta = json.loads(meta_file.read_text(encoding="utf-8"))
            checkpoint_matches = (stored_meta == expected_meta)
        except Exception:
            checkpoint_matches = False

    if checkpoint_matches:
        with open(s2_file, "r", encoding="utf-8") as f:
            S2_by_model[model_name] = json.load(f)
        hist = pd.read_csv(history_file)
        sife_histories.append(hist)
        print(f"Loaded matching S2 checkpoint for {model_name}: {len(S2_by_model[model_name])} features")
        continue
    elif s2_file.exists():
        print(f"Checkpoint for {model_name} exists but config differs; recomputing S2.")

    selected, history_df = sife_python_port(
        estimator=estimator,
        X=X_train,
        y=y_train,
        feature_names=feature_columns,
        model_name=model_name,
        population_size=SIFE_POPULATION,
        max_iter=SIFE_ITERATIONS,
        pc=0.6,
        pm=0.5,
        w1=0.80,
        w2=0.20,
        max_eval_rows=SIFE_EVAL_MAX_ROWS,
        random_state=RANDOM_SEED,
    )
    S2_by_model[model_name] = selected
    sife_histories.append(history_df)

    with open(s2_file, "w", encoding="utf-8") as f:
        json.dump(selected, f, ensure_ascii=False, indent=2)
    history_df.to_csv(history_file, index=False)
    with open(meta_file, "w", encoding="utf-8") as f:
        json.dump(expected_meta, f, ensure_ascii=False, indent=2)
    joblib.dump({"selected_features": selected, "history": history_df, "meta": expected_meta},
                PATHS["checkpoints"] / f"SIFE_checkpoint_{model_name.replace('-', '_')}.joblib")
    print(model_name, "S2:", selected)

sife_history_all = pd.concat(sife_histories, ignore_index=True) if sife_histories else pd.DataFrame()
if not sife_history_all.empty:
    sife_history_all.to_csv(PATHS["feature_selection"] / "SIFE_history_all_models.csv", index=False)

## 17. Tạo hybrid feature subsets S1S2

Theo framework, `S1S2` là hybrid giữa `Ensemble filter` và `Wrapper`: notebook lấy giao `S1 ∩ S2` cho từng model. Nếu giao rỗng, fallback một feature có thứ hạng cao nhất trong `S1`, đồng thời đánh dấu trạng thái fallback để không che giấu kết quả.

In [ ]:
S1S2_by_model = {}
hybrid_status = {}
for model_name, S2 in S2_by_model.items():
    intersection = [f for f in S1 if f in set(S2)]
    fallback_used = False
    if len(intersection) == 0:
        intersection = [S1[0]]
        fallback_used = True
    S1S2_by_model[model_name] = intersection
    hybrid_status[model_name] = {
        "S1_count": len(S1),
        "S2_count": len(S2),
        "S1S2_count": len(intersection),
        "empty_intersection_fallback": fallback_used,
    }

with open(PATHS["feature_selection"] / "S2_all_models.json", "w", encoding="utf-8") as f:
    json.dump(S2_by_model, f, ensure_ascii=False, indent=2)
with open(PATHS["feature_selection"] / "S1S2_all_models.json", "w", encoding="utf-8") as f:
    json.dump(S1S2_by_model, f, ensure_ascii=False, indent=2)
with open(PATHS["feature_selection"] / "hybrid_status.json", "w", encoding="utf-8") as f:
    json.dump(hybrid_status, f, ensure_ascii=False, indent=2)

display(pd.DataFrame(hybrid_status).T)

## 18. Huấn luyện và đánh giá năm ML models trên S0, S1, S2, S1S2

Mọi model được fit trên train và chỉ đánh giá một lần trên test ở bước này. Dùng cùng target, cùng chronological split và cùng định nghĩa metric để đảm bảo so sánh công bằng. `GPR` vẫn dùng giới hạn training size như đã khai báo.

Bài báo báo cáo `RMSE`, `rRMSE`, `MAPE`, `MAE`, `R²`. Công thức `rRMSE` in trong Eq. (21) của bài báo không khớp rõ với thang giá trị trong Table 7; notebook lưu `rRMSE_article_literal` theo biểu thức in ra và thêm `NRMSE_mean_pct` (RMSE chia mean target, %) như một chỉ số kiểm tra bổ sung. Khi viết report, cần nêu rõ điều này, không âm thầm sửa công thức.

In [ ]:
def regression_metrics(y_true, y_pred):
    yt = np.asarray(y_true, dtype=float).ravel()
    yp = np.asarray(y_pred, dtype=float).ravel()
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    mae = float(mean_absolute_error(yt, yp))
    nonzero = np.abs(yt) > 1e-12
    mape_pct = float(np.mean(np.abs((yt[nonzero] - yp[nonzero]) / yt[nonzero])) * 100) if nonzero.any() else np.nan
    r2 = float(r2_score(yt, yp)) if len(yt) > 1 else np.nan
    # Literal reading of Eq. (21) in the article: sqrt(MSE / sum(prediction^2)).
    denom = float(np.sum(yp ** 2))
    rrmse_article_literal = float(np.sqrt(mean_squared_error(yt, yp) / denom)) if denom > 0 else np.nan
    nrmse_mean_pct = float(rmse / max(abs(np.mean(yt)), 1e-12) * 100)
    return {
        "RMSE": rmse,
        "rRMSE_article_literal": rrmse_article_literal,
        "NRMSE_mean_pct_supplement": nrmse_mean_pct,
        "MAPE_pct": mape_pct,
        "MAE": mae,
        "R2": r2,
    }


def model_fit_predict(model, Xtr, ytr, Xte, model_name):
    return _fit_predict_safe(model, Xtr, ytr, Xte, model_name=model_name)

feature_sets = {"S0": {name: feature_columns for name in paper_models},
                "S1": {name: S1 for name in paper_models},
                "S2": S2_by_model,
                "S1S2": S1S2_by_model}

result_rows = []
prediction_frames = []

for subset_name, mapping in feature_sets.items():
    for model_name, estimator in paper_models.items():
        selected_features = mapping[model_name]
        Xtr = X_train[selected_features]
        Xte = X_test[selected_features]
        print(f"Fitting {model_name} | {subset_name} | {len(selected_features)} features")
        y_pred_scaled, fitted_model = model_fit_predict(estimator, Xtr, y_train, Xte, model_name)
        y_pred = scaler_y.inverse_transform(np.asarray(y_pred_scaled).reshape(-1, 1)).ravel()
        metrics = regression_metrics(y_test_raw, y_pred)
        row = {
            "feature_subset": subset_name,
            "model": model_name,
            "n_features": len(selected_features),
            "features": ", ".join(selected_features),
            **metrics,
        }
        result_rows.append(row)
        pred_df = pd.DataFrame({
            "date": time_test.to_numpy(),
            "y_true": y_test_raw.to_numpy(),
            "y_pred": y_pred,
            "residual": y_test_raw.to_numpy() - y_pred,
            "feature_subset": subset_name,
            "model": model_name,
        })
        prediction_frames.append(pred_df)
        # Ghi predictions sau từng model để giảm nguy cơ mất kết quả khi Colab ngắt phiên.
        pd.concat(prediction_frames, ignore_index=True).to_parquet(
            PATHS["metrics"] / "paper_models_predictions_running.parquet", index=False
        )
        model_file = PATHS["models"] / f"{subset_name}_{model_name.replace('-', '_')}.joblib"
        joblib.dump({"model": fitted_model, "features": selected_features, "scaler_y": scaler_y}, model_file)
        pd.DataFrame(result_rows).to_csv(PATHS["metrics"] / "paper_models_running_results.csv", index=False)

paper_results = pd.DataFrame(result_rows)
predictions_all = pd.concat(prediction_frames, ignore_index=True)
paper_results.to_csv(PATHS["metrics"] / "paper_models_results.csv", index=False)
predictions_all.to_parquet(PATHS["metrics"] / "paper_models_predictions.parquet", index=False)

display(paper_results.sort_values(["RMSE", "MAE"]).reset_index(drop=True))

## 19. So sánh các baseline phổ biến trên cùng holdout test

Để biết framework có thực sự có ích hay không, ngoài `S0` của năm model trong bài báo, ta cần các mô hình tham chiếu đơn giản và thông dụng:

- `Persistence`: dự báo bước tiếp theo bằng giá trị hiện tại `Appliances_t`.
- `Ridge`: linear model có regularization.
- `RandomForest`: ensemble cây.
- `HistGradientBoosting`: gradient boosting dựa trên histogram.

Các baseline dùng toàn bộ `S0` features (trừ `Persistence`, vốn chỉ dùng `Appliances_t`) và cùng chronological train/test split. Đây là nhóm so sánh bổ sung do người thực hiện thiết kế, không phải thành phần thuộc bài báo 2024.

In [ ]:
baseline_estimators = {
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=350, min_samples_leaf=2,
                                          random_state=RANDOM_SEED, n_jobs=-1),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=250, learning_rate=0.06,
                                                           max_leaf_nodes=31, random_state=RANDOM_SEED),
}

baseline_rows = []
baseline_predictions = []

# Persistence baseline.
y_pred_persistence = X_test_raw["Appliances_t"].to_numpy(dtype=float)
metrics = regression_metrics(y_test_raw, y_pred_persistence)
baseline_rows.append({"feature_subset": "S0", "model": "Persistence", "n_features": 1,
                      "features": "Appliances_t", **metrics})
baseline_predictions.append(pd.DataFrame({
    "date": time_test.to_numpy(), "y_true": y_test_raw.to_numpy(),
    "y_pred": y_pred_persistence, "residual": y_test_raw.to_numpy() - y_pred_persistence,
    "feature_subset": "S0", "model": "Persistence"
}))

for model_name, estimator in baseline_estimators.items():
    print("Fitting baseline:", model_name)
    fitted = clone(estimator).fit(X_train, y_train)
    pred_scaled = fitted.predict(X_test)
    pred = scaler_y.inverse_transform(np.asarray(pred_scaled).reshape(-1, 1)).ravel()
    metrics = regression_metrics(y_test_raw, pred)
    baseline_rows.append({"feature_subset": "S0", "model": model_name, "n_features": len(feature_columns),
                          "features": ", ".join(feature_columns), **metrics})
    baseline_predictions.append(pd.DataFrame({
        "date": time_test.to_numpy(), "y_true": y_test_raw.to_numpy(),
        "y_pred": pred, "residual": y_test_raw.to_numpy() - pred,
        "feature_subset": "S0", "model": model_name
    }))
    joblib.dump(fitted, PATHS["models"] / f"baseline_{model_name}.joblib")

baseline_results = pd.DataFrame(baseline_rows)
baseline_predictions_all = pd.concat(baseline_predictions, ignore_index=True)
baseline_results.to_csv(PATHS["metrics"] / "baseline_results.csv", index=False)
baseline_predictions_all.to_parquet(PATHS["metrics"] / "baseline_predictions.parquet", index=False)

display(baseline_results.sort_values("RMSE"))

## 20. Bảng tổng hợp chung và xác định model tốt nhất

Kết hợp kết quả từ các feature subsets trong paper framework với baseline phổ biến. RMSE là tiêu chí xếp hạng chính; MAE và R² được dùng để kiểm tra chéo. Không chọn model dựa trên test trong quá trình tuning; bảng này chỉ phục vụ báo cáo kết quả holdout sau khi pipeline đã cố định.

In [ ]:
all_results = pd.concat([paper_results, baseline_results], ignore_index=True)
all_results = all_results.sort_values(["RMSE", "MAE"], ascending=True).reset_index(drop=True)
all_results.to_csv(PATHS["metrics"] / "all_models_comparison.csv", index=False)

paper_avg = paper_results.groupby("feature_subset")[["RMSE", "MAE", "MAPE_pct", "R2"]].mean().sort_values("RMSE")
print("Average performance by paper feature subset:")
display(paper_avg)

print("Overall top 10:")
display(all_results.head(10))

best_row = all_results.iloc[0].to_dict()
with open(PATHS["reports"] / "best_model_summary.json", "w", encoding="utf-8") as f:
    json.dump(best_row, f, ensure_ascii=False, indent=2, default=str)
print("Best configuration:", best_row["feature_subset"], best_row["model"])

## 21. Trực quan hóa kết quả và kiểm tra sai số

Các biểu đồ được lưu trực tiếp vào Google Drive. Khi phân tích, chú ý xem model nào có RMSE/MAE tốt, có hiện tượng lệch có hệ thống hay không, và Feature Selection có giảm số lượng features mà vẫn giữ hiệu năng hay không.

In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(data=all_results.head(15), x="RMSE", y="model", hue="feature_subset", dodge=False)
plt.title("Top configurations theo RMSE trên holdout test")
plt.tight_layout()
plt.savefig(PATHS["figures"] / "top_models_rmse.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 6))
sns.scatterplot(data=paper_results, x="n_features", y="RMSE", hue="feature_subset", style="model", s=90)
plt.title("RMSE và số lượng features – paper framework")
plt.tight_layout()
plt.savefig(PATHS["figures"] / "rmse_vs_number_of_features.png", dpi=160, bbox_inches="tight")
plt.show()

best_key = (best_row["feature_subset"], best_row["model"])
if best_row["model"] == "Persistence":
    best_pred_df = baseline_predictions_all[baseline_predictions_all["model"] == "Persistence"].copy()
elif best_row["feature_subset"] == "S0" and best_row["model"] in baseline_estimators:
    best_pred_df = baseline_predictions_all[baseline_predictions_all["model"] == best_row["model"]].copy()
else:
    best_pred_df = predictions_all[
        (predictions_all["feature_subset"] == best_key[0]) &
        (predictions_all["model"] == best_key[1])
    ].copy()

plt.figure(figsize=(15, 5))
plt.plot(best_pred_df["date"], best_pred_df["y_true"], label="Actual", linewidth=1.2)
plt.plot(best_pred_df["date"], best_pred_df["y_pred"], label="Prediction", linewidth=1.0)
plt.title(f"Actual vs Prediction – {best_key[0]} / {best_key[1]}")
plt.legend()
plt.tight_layout()
plt.savefig(PATHS["figures"] / "best_model_actual_vs_prediction.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10, 5))
sns.histplot(best_pred_df["residual"], bins=50, kde=True)
plt.title("Residual distribution – best configuration")
plt.tight_layout()
plt.savefig(PATHS["figures"] / "best_model_residuals.png", dpi=160, bbox_inches="tight")
plt.show()

## 22. SHAP analysis cho best model

Bài báo sử dụng `SHAP` để giải thích feature contributions của các model tốt nhất. Ở đây ta chọn best configuration theo RMSE holdout, lấy lại đúng feature subset và model đã fit, sau đó tính SHAP trên một background sample và explanation sample có giới hạn để Colab không quá tải. Output của model được inverse-transform để SHAP values biểu diễn theo đơn vị gốc của `Appliances`. SHAP chỉ giải thích model đã huấn luyện; không dùng để thay đổi test set hay điều chỉnh hyperparameters.

In [ ]:
# Tìm lại fitted estimator tương ứng từ artifacts đã lưu.
if best_row["model"] == "Persistence":
    best_model_for_shap = None
    best_features_for_shap = ["Appliances_t"]
elif best_row["model"] in baseline_estimators and best_row["feature_subset"] == "S0":
    best_model_for_shap = joblib.load(PATHS["models"] / f"baseline_{best_row['model']}.joblib")
    best_features_for_shap = feature_columns
else:
    model_path = PATHS["models"] / f"{best_row['feature_subset']}_{best_row['model'].replace('-', '_')}.joblib"
    saved = joblib.load(model_path)
    best_model_for_shap = saved["model"]
    best_features_for_shap = saved["features"]

if best_model_for_shap is None:
    print("Persistence không có model parameters; SHAP không áp dụng cho baseline này.")
else:
    rng = np.random.default_rng(RANDOM_SEED)
    bg_n = min(SHAP_BACKGROUND_ROWS, len(X_train))
    ex_n = min(SHAP_EXPLAIN_ROWS, len(X_test))
    bg_idx = np.sort(rng.choice(len(X_train), size=bg_n, replace=False))
    ex_idx = np.sort(rng.choice(len(X_test), size=ex_n, replace=False))
    X_background = X_train[best_features_for_shap].iloc[bg_idx]
    X_explain = X_test[best_features_for_shap].iloc[ex_idx]

    try:
        masker = shap.maskers.Independent(X_background)
        def predict_original_units(X_input):
            prediction_scaled = best_model_for_shap.predict(np.asarray(X_input, dtype=float))
            return scaler_y.inverse_transform(np.asarray(prediction_scaled).reshape(-1, 1)).ravel()
        explainer = shap.Explainer(predict_original_units, masker)
        shap_values = explainer(X_explain, max_evals=2 * len(best_features_for_shap) + 1)
        shap.summary_plot(shap_values, X_explain, show=False)
        plt.tight_layout()
        plt.savefig(PATHS["figures"] / "shap_summary_best_model.png", dpi=160, bbox_inches="tight")
        plt.show()

        shap_importance = pd.DataFrame({
            "feature": best_features_for_shap,
            "mean_abs_SHAP": np.abs(shap_values.values).mean(axis=0),
        }).sort_values("mean_abs_SHAP", ascending=False)
        display(shap_importance)
        shap_importance.to_csv(PATHS["reports"] / "shap_feature_importance.csv", index=False)
        joblib.dump({"explainer": explainer, "shap_values": shap_values},
                    PATHS["checkpoints"] / "shap_best_model.joblib")
    except Exception as exc:
        print("SHAP calculation failed:", repr(exc))
        print("Các metrics/model vẫn được giữ trong Drive; có thể giảm SHAP_BACKGROUND_ROWS và SHAP_EXPLAIN_ROWS rồi chạy lại cell này.")

## 23. Tổng hợp feature selection và mức giảm số chiều

Bảng này giúp trả lời câu hỏi quan trọng: framework có làm giảm số features so với `S0` hay không, và việc giảm số features có làm thay đổi độ chính xác như thế nào. Kết quả số lượng features cần được đọc cùng RMSE/MAE/R², không kết luận rằng ít features hơn luôn tốt hơn.

In [ ]:
fs_summary_rows = [{"feature_subset": "S0", "model": "All", "n_features": len(feature_columns),
                    "features": ", ".join(feature_columns)}]
fs_summary_rows.append({"feature_subset": "S1", "model": "All", "n_features": len(S1),
                        "features": ", ".join(S1)})
for model_name in paper_models:
    fs_summary_rows.append({"feature_subset": "S2", "model": model_name,
                            "n_features": len(S2_by_model[model_name]),
                            "features": ", ".join(S2_by_model[model_name])})
    fs_summary_rows.append({"feature_subset": "S1S2", "model": model_name,
                            "n_features": len(S1S2_by_model[model_name]),
                            "features": ", ".join(S1S2_by_model[model_name])})
fs_summary = pd.DataFrame(fs_summary_rows)
fs_summary.to_csv(PATHS["reports"] / "feature_subset_summary.csv", index=False)
display(fs_summary)

## 24. Kết luận tự động và checklist phân tích sau tái hiện

Notebook chỉ tạo bảng và tín hiệu định lượng; người làm báo cáo cần diễn giải kết quả, không copy kết luận chung chung. Dùng checklist dưới đây sau khi hoàn thành toàn bộ pipeline:

1. Trong năm model của bài báo, model nào tốt nhất trên dataset hiện tại?
2. `S1`, `S2`, `S1S2` có tốt hơn `S0` không? Chênh lệch RMSE/MAE bao nhiêu phần trăm?
3. `S1S2` có giảm số features so với `S2` hay không? Có mất hiệu năng không?
4. Các baseline `Persistence`, `Ridge`, `RandomForest`, `HistGradientBoosting` có vượt các cấu hình paper-inspired không?
5. `SHAP` cho thấy những sensor/time features nào ảnh hưởng mạnh nhất?
6. Có dấu hiệu overfitting hoặc thay đổi phân phối giữa train và test không?
7. Những hạn chế do dataset 10 phút, phạm vi thời gian ngắn và Python port của `SIFE` ảnh hưởng thế nào đến khả năng so sánh với paper?

**Không** mở rộng thuật toán, thêm DL/Transformer, tuning nâng cao hoặc thay đổi objective trong notebook tái hiện đầu tiên. Những ý tưởng mở rộng chỉ thực hiện sau khi đã lưu và đóng băng kết quả baseline này.

## 25. Sao lưu định kỳ, khôi phục sau khi thoát Google Colab

### Những gì đã được lưu trên Google Drive

- Raw data: bản CSV được đặt trong `data/raw/`.
- Processed data: supervised dataset, feature columns và split metadata trong `data/processed/` và `config/`.
- Model checkpoints: scaler, feature selector, model đã fit, SHAP objects trong `artifacts/checkpoints/` và `artifacts/models/`. Bayesian Optimization và SIFE có metadata để tránh load nhầm checkpoint khác cấu hình.
- Feature selection: scores, `S1`, `S2`, `S1S2`, SIFE history trong `artifacts/feature_selection/`.
- Evaluation: metrics và predictions trong `artifacts/metrics/`.
- Plots và report summaries trong `figures/` và `reports/`.

### Khi quay lại Colab sau khi thoát

1. Mở notebook này trong Google Colab.
2. Chạy cell kết nối Drive ở đầu notebook.
3. Đặt `EXPERIMENT_MODE` giống lần chạy đã lưu trong `config/experiment_config.json`.
4. Không upload CSV lần nữa nếu `data/raw/KAG_energydata_complete.csv` vẫn tồn tại.
5. Các cell có S2 checkpoint sẽ tự load JSON và history nếu cả hai file đã có; các model final đã lưu dạng `joblib`.
6. Nếu runtime reset giữa lúc đang chạy một model, chạy lại cell đó; kết quả đã hoàn thành được ghi ra file sau mỗi model.

### Lưu ý về tính tái lập

Google Drive giữ file qua các phiên, nhưng notebook không tự động khôi phục trạng thái Python đang chạy trong RAM. Muốn tiếp tục chính xác, cần load checkpoint hoặc chạy lại cell tạo biến trong runtime mới. Không xóa thư mục `energy_forecasting_reproduction/` trước khi đã tải backup ZIP.

In [ ]:
# Lưu trạng thái tổng hợp và tạo backup ZIP để tải về hoặc lưu thêm một bản độc lập.
run_manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "experiment_mode": EXPERIMENT_MODE,
    "config": CONFIG,
    "split_info": SPLIT_INFO,
    "dataset_path": str(DATA_PATH),
    "dataset_shape_raw": list(df_raw.shape),
    "dataset_shape_supervised": list(supervised.shape),
    "n_candidate_features": len(feature_columns),
    "S1": S1,
    "S2_by_model": S2_by_model,
    "S1S2_by_model": S1S2_by_model,
    "best_configuration": best_row,
    "notes": [
        "Dataset differs from the UK monthly energy dataset used in the paper.",
        "Target is Appliances(t+1), one 10-minute step ahead.",
        "Relief is adapted to RReliefF for continuous regression target.",
        "SIFE is a documented Python port, not the original MATLAB source implementation.",
        "GPR and LSBoost are scikit-learn approximations of the MATLAB configurations.",
    ],
}
with open(PATHS["reports"] / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(run_manifest, f, ensure_ascii=False, indent=2, default=str)

zip_base = PROJECT_ROOT.parent / f"{PROJECT_NAME}_{EXPERIMENT_MODE}_backup"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=PROJECT_ROOT)
print("Workspace:", PROJECT_ROOT)
print("Backup ZIP:", zip_path)
print("Backup size MB:", round(Path(zip_path).stat().st_size / (1024**2), 2))

## 26. Tóm tắt giới hạn và cách trích dẫn trong báo cáo

Khi viết report, cần trình bày chính xác:

- Đây là **method reproduction trên dataset khác**, không phải replication kết quả gốc.
- Tần suất 10 phút và target `Appliances(t+1)` khác dữ liệu monthly national energy consumption và `EC(t+1)` của bài báo.
- `RReliefF` được dùng vì target liên tục; cần nêu đây là regression variant của Relief.
- `SIFE` được cài dưới dạng Python port dựa trên phần mô tả thuật toán và tham số được công bố. Không nên tuyên bố là mã MATLAB gốc hoặc implementation chính xác từng phép toán nội bộ.
- `GPR` và `LSBoost` dùng ánh xạ scikit-learn; nêu rõ khác biệt so với MATLAB.
- `rRMSE_article_literal` được tính theo biểu thức hiển thị ở Eq. (21); vì công thức và thang giá trị trong bảng kết quả bài báo có dấu hiệu không nhất quán, báo cáo nên nêu cách tính đã dùng và hiển thị thêm `NRMSE_mean_pct_supplement`.
- Không so sánh trực tiếp con số RMSE/MAPE giữa dataset này và Table 7 của bài báo; chỉ so sánh xu hướng tương đối giữa các feature-selection tracks và baseline trên cùng dataset.

### Nguồn chính

1. Eskandari, H., Saadatmand, H., Ramzan, M., & Mousapour, M. (2024). *Innovative framework for accurate and transparent forecasting of energy consumption: A fusion of feature selection and interpretable machine learning*. Applied Energy, 366, 123314. DOI: https://doi.org/10.1016/j.apenergy.2024.123314
2. Saadatmand, H., & Akbarzadeh-T, M.-R. (2023). *Set-based integer-coded fuzzy granular evolutionary algorithms for high-dimensional feature selection*. Applied Soft Computing, 142, 110240. DOI: https://doi.org/10.1016/j.asoc.2023.110240
3. MathWorks documentation, `relieff` (numeric response uses `RReliefF`): https://www.mathworks.com/help/stats/relieff.html

**Notebook kết thúc tại đây.** Chỉ sau khi lưu lại kết quả tái hiện này mới bắt đầu nhánh phân tích và cải tiến riêng.